# 第6章 分类模型 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter6/Chapter6.md]]
- [[课程/深度学习/演示代码/Chapter6.py]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md]]
- [[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md]]（仅分类部分，不含卷积/池化）
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## Q601 四象限标签：零归非负一侧

**演示改编，依据** [[课程/深度学习/PPT/Chapter6/Chapter6.md]]“演示：四象限分类”小节、[[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md#二、训练集与测试集]]。

实现 `quadrant_labels(x)`，输入(N,2)，输出(N,) long，按课程第一至第四象限编号1/2/3/4：(+,+)、(-,+)、(-,-)、(+,-)，零并入≥0一侧。
验证(0,0)、(0,-1)、(-1,0)、(0,1)、(1,0)；生成标准正态512/192独立训练/测试点，取seed601。
定义labels1_601为1—4，定义labels0_601=labels1_601−1并保持long，以供库交叉熵使用。
不要仅用>0遗漏坐标轴，也不要把torch.max的最大值当标签下标。

**自检接口／需保存的结果**

`boundary601`：按题面所列五个轴上点的**原顺序**得到的标签，形状`(5,)`的long Tensor。训练、测试输入分别保存为`xtrain601`、`xtest601`，float32 Tensor形状为`(512,2)`、`(192,2)`；已要求的`labels1_601`和`labels0_601`均对应训练集，形状`(512,)`。


<details><summary>提示（先尝试再展开）</summary>

四个掩码互斥且穷尽。先保留课程1—4编号，明确转换一次，而非在绘图与损失中反复猜加一或减一。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def quadrant_labels(x):
    if x.ndim != 2 or x.shape[1] != 2 or not torch.isfinite(x).all():
        raise ValueError("X must be finite with shape (N,2)")
    y = torch.empty(len(x), dtype=torch.long)
    y[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    y[(x[:,0]<0)&(x[:,1]>=0)] = 2
    y[(x[:,0]<0)&(x[:,1]<0)] = 3
    y[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return y

class Classifier6(nn.Module):
    def __init__(self, in_dim=2, classes=4, depth=3, width=10, activation="ReLU"):
        super().__init__()
        if depth < 2:
            raise ValueError("depth >= 2")
        activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
        layers, previous = [], in_dim
        for _ in range(depth-1):
            layers += [nn.Linear(previous,width), activations[activation]()]
            previous = width
        self.features = nn.Sequential(*layers)
        self.output = nn.Linear(width, classes)
    def forward(self, x):
        return self.output(self.features(x))  # raw logits, NO softmax here

def fit_quadrants6(steps=100, seed=607):
    torch.manual_seed(seed)
    xt, xs = torch.randn(512,2), torch.randn(192,2)
    yt, ys = quadrant_labels(xt)-1, quadrant_labels(xs)-1
    net = Classifier6(width=16)
    optimizer = torch.optim.Adam(net.parameters(),lr=.02)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(xt),yt)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        logits = net(xs)
        pred0 = logits.argmax(1)
        train_loss = loss_fn(net(xt),yt).item()
        train_acc = (net(xt).argmax(1)==yt).float().mean().item()
        test_acc = (pred0==ys).float().mean().item()
    report = dict(train_loss=train_loss,train_accuracy=train_acc,test_accuracy=test_acc)
    return net, history, (xt,yt,xs,ys), report

torch.manual_seed(601)
points601 = torch.tensor([[0.,0.],[0.,-1.],[-1.,0.],[0.,1.],[1.,0.]])
boundary601 = quadrant_labels(points601)
xtrain601,xtest601 = torch.randn(512,2),torch.randn(192,2)
labels1_601 = quadrant_labels(xtrain601)
labels0_601 = labels1_601-1
print("Axis labels:",boundary601.tolist(),"CE targets:",(boundary601-1).tolist())


### 解析与点评

轴上点的归属由题目规则确定，不是数学“坐标轴不属于象限”的一般措辞。PyTorch整数目标必须0到J−1且long；课堂原脚本浮点1—4只用于one-hot及比较，不能直接当整数CE目标。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert boundary601.tolist() == [1,4,2,1,1]
assert quadrant_labels(torch.tensor([[1.,1.],[-1.,1.],[-1.,-1.],[1.,-1.]])).tolist() == [1,2,3,4]
assert labels0_601.dtype == torch.long
assert 0 <= labels0_601.min() and labels0_601.max() <= 3
assert quadrant_labels(torch.empty(0,2)).shape == (0,)
assert xtrain601.shape == (512,2) and xtest601.shape == (192,2)

print("自检通过")


## Q602 one-hot矩阵：编号、列下标与dtype

**新编/作业汇编改编，依据** [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C6-06 从逻辑掩码到one-hot矩阵]]、[[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md#四、把类别标签转换为one-hot向量]]。

实现 `one_hot_1based(labels,J)`，输入(N,)、数值为1—J的Tensor（允许整数值浮点，如原演示），返回float32的(N,J)矩阵。
用按类别循环布尔赋值实现，不用额外下载或数据集；拒绝0、J+1、1.5等非整数标签。空输入返回(0,J)。
对[4,2,1,3]构造onehot602；用argmax+1逆变换，说明如果标签原本是0—9则不应减一。
补充API可对照 `torch.nn.functional.one_hot(labels.long()-1,num_classes=J).float()`，但要先检查范围与整数性。


<details><summary>提示（先尝试再展开）</summary>

类别j写到列j−1，shape与dtype都要检查。不能先long再检查1.5，否则会无声截断。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def one_hot_1based(labels, J):
    if labels.ndim != 1 or J < 1:
        raise ValueError("vector labels and J>=1 required")
    if (not torch.isfinite(labels).all() or not torch.equal(labels,labels.round())
            or (labels.numel() and (labels.min()<1 or labels.max()>J))):
        raise ValueError("labels must be integer-valued 1..J")
    result = torch.zeros(len(labels),J,dtype=torch.float32)
    for j in range(1,J+1):
        result[labels==j,j-1] = 1
    return result

onehot602 = one_hot_1based(torch.tensor([4.,2.,1.,3.]),4)
print(onehot602)


### 解析与点评

one-hot每行和为1，但该性质本身不证明列号正确，因此自检也逆变换。整数目标CE与浮点概率目标CE是两种标签接口，不应把long one-hot误作二者任意一种。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert onehot602.shape == (4,4) and onehot602.dtype == torch.float32
assert torch.equal(onehot602.sum(1),torch.ones(4))
assert (onehot602.argmax(1)+1).tolist() == [4,2,1,3]
assert one_hot_1based(torch.empty(0),4).shape == (0,4)
for bad602 in (torch.tensor([0]),torch.tensor([5]),torch.tensor([1.5])):
    try:
        one_hot_1based(bad602,4)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid labels accepted")

print("自检通过")


## Q603 逐行稳定Softmax与平移不变性

**新编，依据** [[课程/深度学习/PPT/Chapter6/Chapter6.md]]“Softmax函数”小节、[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C6-04 Softmax作用在什么方向]]。

实现NumPy `stable_softmax6(z)`：输入有限二维float数组(N,J)、J≥1，逐行先减最大值，再exp并除以行和，输出同形概率；为简化题面要求N≥1。
对[[1000,999,-1000],[1,1,1]]与每行加常数的版本检查；报告softmax603。
解释dim=1/axis=1归一化类别，axis=0是按同一类跨样本，不能作为每个样本的概率向量。
不要先对巨大正logits求exp，再想办法修复inf。

**自检接口／需保存的结果**

将题面两行原始logits保存为`logits603`，NumPy浮点数组，形状`(2,3)`；已要求的`softmax603`为该数组逐行归一化后的NumPy数组，形状同为`(2,3)`。不要用每行加常数后的数组覆盖原始`logits603`。


<details><summary>提示（先尝试再展开）</summary>

共同减去行最大值不改指数比。keepdims保留列维以便按行广播。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def stable_softmax6(z):
    z = np.asarray(z,dtype=np.float64)
    if z.ndim != 2 or min(z.shape)<1 or not np.isfinite(z).all():
        raise ValueError("finite nonempty 2D logits required")
    shifted = z-z.max(axis=1,keepdims=True)
    exp = np.exp(shifted)
    return exp/exp.sum(axis=1,keepdims=True)

logits603 = np.array([[1000.,999.,-1000.],[1.,1.,1.]])
softmax603 = stable_softmax6(logits603)
print(softmax603)


### 解析与点评

稳定Softmax避免上溢，但极小概率仍可能下溢到0；因此交叉熵更应直接用稳定log-softmax，而非对softmax结果盲目log。argmax对同一行的Softmax不变，分类预测可直接取logits.argmax。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.isfinite(softmax603).all() and np.allclose(softmax603.sum(1),1)
assert np.allclose(softmax603[1],np.ones(3)/3)
assert np.allclose(stable_softmax6(logits603+np.array([[50.],[-500.]])),softmax603)
assert np.isclose(softmax603[0,0],1/(1+np.exp(-1)))
assert stable_softmax6([[7.]]).item() == 1
assert np.array_equal(softmax603.argmax(1), logits603.argmax(1))

print("自检通过")


## Q604 概率接口的手动交叉熵：避免0乘log0

**新编，依据** [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C6-03 为什么使用交叉熵]]、[[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md#自测（新编）：损失计算的维度]]。

实现 `probability_ce6(p,y0)`：p=(N,J)非负有限、每行和为1；y0=(N,)合法0起始整数类别；返回真实类别概率的负对数均值float。
只取真实类别概率，避免其他分量0导致0*log(0)成为NaN。真实类别概率为0时返回+inf，不通过clip伪造有限值。
例：两行[.7,.1,.1,.1]、[.1,.2,.6,.1]，y0=[0,2]，ce604≈.43375。
另算[.2,.6,.2]和[0,.6,.4]的CE及对one-hot(0,1,0)的平方和，展示相同CE而平方和不同，不能推论所有分类中MSE都无效。

**自检接口／需保存的结果**

已要求的`ce604`为前述两样本批量交叉熵（Python float）。`pair_ce604`按后述两组预测的题面顺序，保存各自对真实类别1（0起始）的单样本交叉熵，长度2的Python float列表；`squared604`按相同顺序保存对one-hot标签的**各行误差平方和**，NumPy浮点数组，形状`(2,)`，不是对三个分量再求均值。


<details><summary>提示（先尝试再展开）</summary>

one-hot标签仅选中真实类别；这允许用整数索引避免计算无关的log0。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def probability_ce6(p,y0):
    p = np.asarray(p,dtype=float)
    y = np.asarray(y0)
    if p.ndim!=2 or len(p)==0 or p.shape[1]==0 or y.shape!=(len(p),):
        raise ValueError("bad shapes")
    if not np.isfinite(p).all() or (p<0).any() or not np.allclose(p.sum(1),1):
        raise ValueError("invalid probabilities")
    if not np.issubdtype(y.dtype,np.integer) or (y<0).any() or (y>=p.shape[1]).any():
        raise ValueError("0-based integer labels required")
    true_prob = p[np.arange(len(p)),y]
    if (true_prob==0).any():
        return float("inf")
    return float(-np.log(true_prob).mean())

p604 = np.array([[.7,.1,.1,.1],[.1,.2,.6,.1]])
ce604 = probability_ce6(p604,np.array([0,2]))
q604 = np.array([[.2,.6,.2],[0.,.6,.4]])
pair_ce604 = [probability_ce6(row[None,:],np.array([1])) for row in q604]
squared604 = ((q604-np.array([0.,1.,0.]))**2).sum(1)
print("Batch CE:",ce604,"Equal CE:",pair_ce604,"Squared errors:",squared604)


### 解析与点评

形状从(N,J)逐元素乘后仍(N,J)，沿类求和变(N,)，对样本mean变标量。准确率只看最大值位置；交叉熵还看真实类置信度，因此两指标不能互相替代。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.isclose(ce604,-np.log(.42)/2)
assert np.allclose(pair_ce604,[-np.log(.6)]*2)
assert np.allclose(squared604,[.24,.32])
assert probability_ce6([[0.,1.]],np.array([1])) == 0
assert np.isinf(probability_ce6([[0.,1.]],np.array([0])))
try:
    probability_ce6([[.2,.3]],np.array([0]))
except ValueError:
    pass
else:
    raise AssertionError("non-normalized probabilities accepted")

print("自检通过")


## Q605 稳定logits交叉熵与库接口核验

**新编/课堂补充改编，依据** [[课程/深度学习/PPT/Chapter6/Chapter6.md]]“Softmax结合交叉熵的计算”小节、[[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md#从概率到稳定交叉熵（第六章补充解释）]]。

实现torch函数 `stable_ce6(z,y0)`：z=(N,J)logits、y0=(N,)long，返回可反传零维Tensor；按行减max，计算logsum(exp(shifted))−shifted_true后mean，不先softmax再log。
用[[1000,999],[-1000,1000]]，真实类[0,0]验证有限损失；第二样本是高置信错分，应有约2000的损失而非inf。
用相同输入比较nn.CrossEntropyLoss，分别backward比较grad605_manual与grad605_library。
解释：库函数输入**raw logits**，不要二次softmax。参考[官方接口](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)；这是API校正，不冒称原Chapter6.py已使用此接口。

**自检接口／需保存的结果**

- `z605`：题面两行logits，CPU **float64** Tensor，形状`(2,2)`、允许求导；`y605`：对应0起始真实类别，long Tensor，形状`(2,)`。
- `manual605`、`library605`：分别为`stable_ce6`与`nn.CrossEntropyLoss`在上述输入上的返回值，均为零维、float64、可反传Tensor；保留Tensor，不要用`.item()`结果替代。
- 已要求的`grad605_manual`、`grad605_library`：各方法独立求导后保存的logits梯度副本，float64 Tensor形状均为`(2,2)`，不相互累加或覆盖。


<details><summary>提示（先尝试再展开）</summary>

移位后最大exp=1，分母至少1。真实类偏移值可能为−2000；直接减它仍保持有限。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def stable_ce6(z,y0):
    if z.ndim!=2 or z.shape[0]==0 or y0.shape!=(z.shape[0],) or y0.dtype!=torch.long:
        raise ValueError("logits (N,J), long target (N,) required")
    if z.shape[1]<1 or not torch.isfinite(z).all() or y0.min()<0 or y0.max()>=z.shape[1]:
        raise ValueError("invalid values")
    shifted = z-z.max(dim=1,keepdim=True).values
    log_sum = shifted.exp().sum(1).log()
    selected = shifted[torch.arange(len(z)),y0]
    return (log_sum-selected).mean()

z605 = torch.tensor([[1000.,999.],[-1000.,1000.]],dtype=torch.float64,requires_grad=True)
y605 = torch.tensor([0,0],dtype=torch.long)
manual605 = stable_ce6(z605,y605)
manual605.backward()
grad605_manual = z605.grad.detach().clone()
z605.grad = None
library605 = nn.CrossEntropyLoss()(z605,y605)
library605.backward()
grad605_library = z605.grad.detach().clone()
print("Manual/library CE:",manual605.item(),library605.item())


### 解析与点评

原Chapter6.py采用显式softmax+onehot手动log，非CrossEntropyLoss。两种数学模型可对应，但函数输入约定不可混用。若把概率送入CE，库会把0到1的概率重新当logits，既改变损失也改变梯度。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert manual605.ndim == 0 and torch.isfinite(manual605)
assert torch.allclose(manual605,library605)
assert torch.allclose(grad605_manual,grad605_library,atol=1e-12)
assert np.isclose(manual605.item(),(np.log1p(np.exp(-1))+2000)/2)
assert torch.allclose(stable_ce6(z605+300,y605),manual605)
assert torch.allclose(stable_ce6(torch.zeros(1,4),torch.tensor([2])),torch.tensor(np.log(4),dtype=torch.float32))

print("自检通过")


## Q606 分类网络输出logits：层注册与参数量

**演示改编，依据** [[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md#三、深度可调的全连接网络]]、[[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md#Chapter6.py逐段对照与运行边界]]。

实现自定义 `Classifier6(in_dim=2,classes=4,depth=3,width=10,activation="ReLU")`，depth计Linear层。隐藏层可用注册的Sequential，末层不加Softmax。
支持深度3、5并检查参数184、404，支持batch1输出(1,4)；概率在评估时由torch.softmax(logits,dim=1)得到。
命名net606、logits606、prob606；使用CrossEntropyLoss和0—3 long标签做一次反传，确认所有参数有grad。
补充说明：任意深度是练习扩展，原my_net_class只分L=3/5；普通Python list不注册层，Sequential/ModuleList会注册。移除原默认cuda设置，CPU可跑，不修改原文件。

**自检接口／需保存的结果**

`counts606`为按depth=3、depth=5顺序保存的两个参数总数的Python整数列表（in_dim=2、classes=4、width=10）。已要求的`net606`使用默认depth=3配置；供一次反传使用的批量含6个二维样本，因此`logits606`、`prob606`均保存为形状`(6,4)`的float32 Tensor，前者是raw logits，后者是对应逐行概率。


<details><summary>提示（先尝试再展开）</summary>

30+110+44=184；四类输出不是一个类别数值。概率转换与训练损失接口分开。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def quadrant_labels(x):
    if x.ndim != 2 or x.shape[1] != 2 or not torch.isfinite(x).all():
        raise ValueError("X must be finite with shape (N,2)")
    y = torch.empty(len(x), dtype=torch.long)
    y[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    y[(x[:,0]<0)&(x[:,1]>=0)] = 2
    y[(x[:,0]<0)&(x[:,1]<0)] = 3
    y[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return y

class Classifier6(nn.Module):
    def __init__(self, in_dim=2, classes=4, depth=3, width=10, activation="ReLU"):
        super().__init__()
        if depth < 2:
            raise ValueError("depth >= 2")
        activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
        layers, previous = [], in_dim
        for _ in range(depth-1):
            layers += [nn.Linear(previous,width), activations[activation]()]
            previous = width
        self.features = nn.Sequential(*layers)
        self.output = nn.Linear(width, classes)
    def forward(self, x):
        return self.output(self.features(x))  # raw logits, NO softmax here

def fit_quadrants6(steps=100, seed=607):
    torch.manual_seed(seed)
    xt, xs = torch.randn(512,2), torch.randn(192,2)
    yt, ys = quadrant_labels(xt)-1, quadrant_labels(xs)-1
    net = Classifier6(width=16)
    optimizer = torch.optim.Adam(net.parameters(),lr=.02)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(xt),yt)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        logits = net(xs)
        pred0 = logits.argmax(1)
        train_loss = loss_fn(net(xt),yt).item()
        train_acc = (net(xt).argmax(1)==yt).float().mean().item()
        test_acc = (pred0==ys).float().mean().item()
    report = dict(train_loss=train_loss,train_accuracy=train_acc,test_accuracy=test_acc)
    return net, history, (xt,yt,xs,ys), report

torch.manual_seed(606)
net606 = Classifier6()
logits606 = net606(torch.randn(6,2))
prob606 = torch.softmax(logits606,dim=1)
loss606 = nn.CrossEntropyLoss()(logits606,torch.tensor([0,1,2,3,0,1]))
loss606.backward()
counts606 = [sum(p.numel() for p in Classifier6(depth=d).parameters()) for d in (3,5)]
print("Parameter counts:",counts606,"Probability row sums:",prob606.sum(1))


### 解析与点评

注册层使parameters、state_dict及优化器都能找到这些张量。全连接网络仍然只做Linear/ReLU，不混入第7章CNN。判断是否正确不能只查输出和≈1，因为本题训练输出本来就应是非归一化logits。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert counts606 == [184,404]
assert logits606.shape == prob606.shape == (6,4)
assert torch.allclose(prob606.sum(1),torch.ones(6))
assert net606(torch.zeros(1,2)).shape == (1,4)
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in net606.parameters())
assert not any(isinstance(m,nn.Softmax) for m in net606.modules())

print("自检通过")


## Q607 四象限完整训练、预测与准确率

**演示改编，依据** [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C6-05 四象限分类的完整实验]]、[[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md#二、训练循环为什么要重算损失（核心）]]。

实现 `fit_quadrants6(steps=100,seed=607)`，固定seed标准正态训练512/测试192点，标签转0—3 long。
网络2→16→16→4，ReLU隐藏、raw logits输出，Adam(.02)100步全批，CrossEntropyLoss。
返回(net,history,(x_train,y0_train,x_test,y0_test),report)，report含最终train_loss/train_accuracy/test_accuracy。loss每步重算，评估no_grad。
打印与原演示的差异：原10000/1000、宽10、5000步、低学习率线性衰减且默认cuda；本地轻量CPU版不保证相同准确率。
有限样本初始准确率不必恰好25%；不要把测试准确率加入可反传loss。

**自检接口／需保存的结果**

将默认调用返回的四项分别保存为`net607`、`history607`、`data607`、`report607`。其中`history607`是100个更新前CE组成的Python float列表；`data607`是按`(x_train,y0_train,x_test,y0_test)`排列的四元组，形状依次为`(512,2)`、`(512,)`、`(192,2)`、`(192,)`，两个标签Tensor均为0—3 long。

`report607`为题面约定的三键Python字典（`train_loss`、`train_accuracy`、`test_accuracy`），值为最终评估的Python float；准确率使用0—1比例而非0—100百分数。


<details><summary>提示（先尝试再展开）</summary>

准确率为正确数/N，选argmax位置；如果拿1—4原标签比较则预测要+1。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def quadrant_labels(x):
    if x.ndim != 2 or x.shape[1] != 2 or not torch.isfinite(x).all():
        raise ValueError("X must be finite with shape (N,2)")
    y = torch.empty(len(x), dtype=torch.long)
    y[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    y[(x[:,0]<0)&(x[:,1]>=0)] = 2
    y[(x[:,0]<0)&(x[:,1]<0)] = 3
    y[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return y

class Classifier6(nn.Module):
    def __init__(self, in_dim=2, classes=4, depth=3, width=10, activation="ReLU"):
        super().__init__()
        if depth < 2:
            raise ValueError("depth >= 2")
        activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
        layers, previous = [], in_dim
        for _ in range(depth-1):
            layers += [nn.Linear(previous,width), activations[activation]()]
            previous = width
        self.features = nn.Sequential(*layers)
        self.output = nn.Linear(width, classes)
    def forward(self, x):
        return self.output(self.features(x))  # raw logits, NO softmax here

def fit_quadrants6(steps=100, seed=607):
    torch.manual_seed(seed)
    xt, xs = torch.randn(512,2), torch.randn(192,2)
    yt, ys = quadrant_labels(xt)-1, quadrant_labels(xs)-1
    net = Classifier6(width=16)
    optimizer = torch.optim.Adam(net.parameters(),lr=.02)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(xt),yt)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        logits = net(xs)
        pred0 = logits.argmax(1)
        train_loss = loss_fn(net(xt),yt).item()
        train_acc = (net(xt).argmax(1)==yt).float().mean().item()
        test_acc = (pred0==ys).float().mean().item()
    report = dict(train_loss=train_loss,train_accuracy=train_acc,test_accuracy=test_acc)
    return net, history, (xt,yt,xs,ys), report

net607,history607,data607,report607 = fit_quadrants6()
print(report607)
print("CPU adaptation: 512/192 samples, width 16, 100 steps; not original full-scale run")


### 解析与点评

测试中0.45仅说明程序没有完全失效，不是课程作业的合格线。保留每一步新前向计算，不能重复最初loss.backward()，也不需要retain_graph=True来掩盖旧图复用。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert len(history607) == 100 and np.isfinite(history607).all()
assert data607[0].shape == (512,2) and data607[2].shape == (192,2)
assert data607[1].dtype == torch.long
assert 0 <= report607["train_accuracy"] <= 1 and 0 <= report607["test_accuracy"] <= 1
assert report607["test_accuracy"] >= .45  # deliberately loose smoke check, not grading target
with torch.no_grad():
    expected607 = (net607(data607[2]).argmax(1)==data607[3]).float().mean().item()
assert np.isclose(expected607,report607["test_accuracy"])

print("自检通过")


## Q608 按预测类别上色：损失曲线与分类图

**演示改编，依据** [[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md#三、从概率矩阵得到准确率与动画（核心）]]、[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C6-08 预测、准确率与训练动画]]。

独立重跑Q607的轻量四象限训练，取seed608、100步。用no_grad得到pred608=logits.argmax(1)+1及概率。
保存output/Q608.png（三面板）：训练CE曲线、测试真实标签着色、测试**预测**标签着色。两张分类图相同坐标范围/颜色映射，标出坐标轴，英文标题明确True/Predicted。
用detach().cpu().numpy()转换；不调用sleep、不依赖交互GUI。输出错误个数与准确率，说明按真标签着色不能展示模型学到的分区。
本地文件路径不同于正式题指定的/output/Qxxx.png，正式环境必须服从题面。

**自检接口／需保存的结果**

- `points608`：用于绘图的同一批测试点，有限NumPy浮点数组，形状`(192,2)`；`prob608`：相应测试概率的float32 Tensor，形状`(192,4)`。已要求的`pred608`为1—4 long Tensor，形状`(192,)`。
- `errors608`：这192条测试数据的错分个数，Python int；`report608`为本次seed608训练的最终报告字典，键为`train_loss`、`train_accuracy`、`test_accuracy`，值为Python float、准确率取0—1比例。报告与预测、错误数必须来自同一次训练和同一测试集。


<details><summary>提示（先尝试再展开）</summary>

输出概率时用dim=1；预测与真标签使用同一1起始约定。所有图均用相同颜色表示同一类别。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def quadrant_labels(x):
    if x.ndim != 2 or x.shape[1] != 2 or not torch.isfinite(x).all():
        raise ValueError("X must be finite with shape (N,2)")
    y = torch.empty(len(x), dtype=torch.long)
    y[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    y[(x[:,0]<0)&(x[:,1]>=0)] = 2
    y[(x[:,0]<0)&(x[:,1]<0)] = 3
    y[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return y

class Classifier6(nn.Module):
    def __init__(self, in_dim=2, classes=4, depth=3, width=10, activation="ReLU"):
        super().__init__()
        if depth < 2:
            raise ValueError("depth >= 2")
        activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
        layers, previous = [], in_dim
        for _ in range(depth-1):
            layers += [nn.Linear(previous,width), activations[activation]()]
            previous = width
        self.features = nn.Sequential(*layers)
        self.output = nn.Linear(width, classes)
    def forward(self, x):
        return self.output(self.features(x))  # raw logits, NO softmax here

def fit_quadrants6(steps=100, seed=607):
    torch.manual_seed(seed)
    xt, xs = torch.randn(512,2), torch.randn(192,2)
    yt, ys = quadrant_labels(xt)-1, quadrant_labels(xs)-1
    net = Classifier6(width=16)
    optimizer = torch.optim.Adam(net.parameters(),lr=.02)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(xt),yt)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        logits = net(xs)
        pred0 = logits.argmax(1)
        train_loss = loss_fn(net(xt),yt).item()
        train_acc = (net(xt).argmax(1)==yt).float().mean().item()
        test_acc = (pred0==ys).float().mean().item()
    report = dict(train_loss=train_loss,train_accuracy=train_acc,test_accuracy=test_acc)
    return net, history, (xt,yt,xs,ys), report

net608,history608,data608,report608 = fit_quadrants6(seed=608)
xt608,yt608,xs608,ys608 = data608
with torch.no_grad():
    pred608 = net608(xs608).argmax(1)+1
    prob608 = torch.softmax(net608(xs608),dim=1)
points608 = xs608.detach().cpu().numpy()
Path("output").mkdir(exist_ok=True)
fig,axes = plt.subplots(1,3,figsize=(12,3.7))
axes[0].plot(history608)
axes[0].set(xlabel="Update",ylabel="Training CE",title="Training loss")
colors608 = ["red","green","blue","orange"]
for ax,labels,title in [(axes[1],ys608+1,"True classes"),(axes[2],pred608,"Predicted classes")]:
    lab = labels.detach().cpu().numpy()
    for j,color in enumerate(colors608,1):
        mask = lab==j
        ax.scatter(points608[mask,0],points608[mask,1],s=7,c=color,label=str(j))
    ax.axhline(0,color="gray",lw=.5)
    ax.axvline(0,color="gray",lw=.5)
    ax.set(xlabel="x1",ylabel="x2",title=title,xlim=(-3.5,3.5),ylim=(-3.5,3.5))
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig("output/Q608.png",dpi=120)
plt.close(fig)
errors608 = int((pred608!=(ys608+1)).sum())
print("Errors:",errors608,"Accuracy:",report608["test_accuracy"])


### 解析与点评

分类图容易揭示编号错位、轴边界和分区学错；准确率只能给一个比例。detach不是“清空整个计算图”，cpu是设备转换，numpy是数据接口转换。此处保存静态图替代原脚本动画以便离线练习。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert pred608.shape == (192,) and 1 <= pred608.min() <= pred608.max() <= 4
assert torch.allclose(prob608.sum(1),torch.ones(192),atol=1e-6)
assert np.isclose(1-errors608/192,report608["test_accuracy"],atol=1e-7)
assert Path("output/Q608.png").stat().st_size > 1000
assert np.isfinite(points608).all()

print("自检通过")


## Q609 Homework03 Exercise 2：40特征10类接口练习

**作业改编，精确来源** [[课程/深度学习/作业/Homework03/Homework03.md]] Exercise 2 (1)—(2)。

**资料缺口**：原hw03第二份.data未找到，因此**未核验实际文件的输入维数与内容**。原作业是MNIST1D、训练/测试4000/1000；本题明确新编40特征10类合成信号替代接口，**不是MNIST1D**，不代表其难度、分布或性能。

实现 `load_classification_data(path=None,*,trusted_pickle=False,n_train=320,n_test=120,seed=609)`：默认生成40个等距t∈[0,2π]，类别k的基准信号sin((k+1)t)+.3cos((k+2)t)，加.15高斯噪声；标签0—9近似均衡并打乱。返回四个数据键及origin，打印来源。
现有本地pickle只在明确trusted_pickle=True时读取，校验键x_train/y_train/x_test/y_test、X=(N,40)、有限、标签整数0—9、长度一致。若实际可信原文件维数不同，应据核验结果调整模型，不能强称它就是40。

实现 `fit_classification(data,depth=3,width=100,activation="ReLU",optimizer_name="SGD",lr=.03,steps=100,seed=609)`，40→100→100→10、raw logits+CrossEntropyLoss、SGD全批100步。
输出最终train_loss、test_accuracy，返回(net,history,report)。显示概率才softmax；不把作业数学softmax输出与库logits输入混用。

**自检接口／需保存的结果**

- 自检先使用默认离线合成路径，将返回字典保存为`data609`：`x_train`、`x_test`为float32 Tensor，形状`(320,40)`、`(120,40)`；`y_train`、`y_test`为0—9 long Tensor，形状`(320,)`、`(120,)`。`origin`为来源字符串，须同时包含大写`SYNTHETIC`与`NOT MNIST1D`，不能将替代数据写成原数据。
- 将默认基准拟合返回的网络、100个更新前CE组成的Python float列表、最终报告字典分别保存为`net609`、`history609`、`report609`。网络输出`(N,10)` raw logits。
- `report609`约定键为`train_loss`、`test_accuracy`（Python float，准确率为0—1比例）及`parameter_count`（Python int）。可信原文件的另行试验不覆盖默认自检结果。


<details><summary>提示（先尝试再展开）</summary>

十类标签原本0—9，不减一。类别损失的输入为logits，标签long。原作业理论softmax模型与库内部log-softmax相对应。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def quadrant_labels(x):
    if x.ndim != 2 or x.shape[1] != 2 or not torch.isfinite(x).all():
        raise ValueError("X must be finite with shape (N,2)")
    y = torch.empty(len(x), dtype=torch.long)
    y[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    y[(x[:,0]<0)&(x[:,1]>=0)] = 2
    y[(x[:,0]<0)&(x[:,1]<0)] = 3
    y[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return y

class Classifier6(nn.Module):
    def __init__(self, in_dim=2, classes=4, depth=3, width=10, activation="ReLU"):
        super().__init__()
        if depth < 2:
            raise ValueError("depth >= 2")
        activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
        layers, previous = [], in_dim
        for _ in range(depth-1):
            layers += [nn.Linear(previous,width), activations[activation]()]
            previous = width
        self.features = nn.Sequential(*layers)
        self.output = nn.Linear(width, classes)
    def forward(self, x):
        return self.output(self.features(x))  # raw logits, NO softmax here

def fit_quadrants6(steps=100, seed=607):
    torch.manual_seed(seed)
    xt, xs = torch.randn(512,2), torch.randn(192,2)
    yt, ys = quadrant_labels(xt)-1, quadrant_labels(xs)-1
    net = Classifier6(width=16)
    optimizer = torch.optim.Adam(net.parameters(),lr=.02)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(xt),yt)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        logits = net(xs)
        pred0 = logits.argmax(1)
        train_loss = loss_fn(net(xt),yt).item()
        train_acc = (net(xt).argmax(1)==yt).float().mean().item()
        test_acc = (pred0==ys).float().mean().item()
    report = dict(train_loss=train_loss,train_accuracy=train_acc,test_accuracy=test_acc)
    return net, history, (xt,yt,xs,ys), report

def load_classification_data(path=None, *, trusted_pickle=False, n_train=320, n_test=120, seed=609):
    if path is not None and Path(path).is_file():
        if not trusted_pickle:
            raise ValueError("pickle can execute code; explicit trusted opt-in required")
        import pickle
        with Path(path).open("rb") as f:
            raw = pickle.load(f)
        origin = "TRUSTED local pickle: " + str(Path(path).resolve())
    else:
        g = torch.Generator().manual_seed(seed)
        t = torch.linspace(0, 2*np.pi, 40)
        classes = torch.arange(10).reshape(-1,1)
        prototypes = torch.sin((classes+1)*t)+.3*torch.cos((classes+2)*t)
        def sample(n):
            y = torch.arange(n) % 10
            y = y[torch.randperm(n,generator=g)]
            x = prototypes[y] + .15*torch.randn(n,40,generator=g)
            return x, y
        xt,yt = sample(n_train)
        xs,ys = sample(n_test)
        raw = dict(x_train=xt,y_train=yt,x_test=xs,y_test=ys)
        origin = "SYNTHETIC 40-feature/10-class signals; NOT MNIST1D or original homework .data"
    data = {}
    for split in ("train","test"):
        x = torch.as_tensor(raw["x_"+split],dtype=torch.float32).clone()
        raw_y = torch.as_tensor(raw["y_"+split]).reshape(-1)
        if not torch.isfinite(raw_y).all() or not torch.equal(raw_y,raw_y.round()):
            raise ValueError("integer-valued finite labels required")
        y = raw_y.long().clone()
        if x.ndim != 2 or x.shape[1] != 40 or len(x) != len(y) or len(y) == 0:
            raise ValueError("expected X=(N,40), y=(N,)")
        if not torch.isfinite(x).all() or y.min()<0 or y.max()>9:
            raise ValueError("finite features, labels 0..9 required")
        data["x_"+split],data["y_"+split] = x,y
    data["origin"] = origin
    print("Data source:",origin)
    return data

def fit_classification(data, *, depth=3, width=100, activation="ReLU", optimizer_name="SGD",
                       lr=.03, steps=100, seed=609):
    if steps < 1:
        raise ValueError("steps > 0 required")
    torch.manual_seed(seed)
    net = Classifier6(40,10,depth,width,activation)
    optimizers = {"SGD":torch.optim.SGD,"Adagrad":torch.optim.Adagrad,
                  "RMSprop":torch.optim.RMSprop,"Adam":torch.optim.Adam}
    optimizer = optimizers[optimizer_name](net.parameters(),lr=lr)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(net(data["x_train"]),data["y_train"])
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        train_loss = loss_fn(net(data["x_train"]),data["y_train"]).item()
        test_accuracy = (net(data["x_test"]).argmax(1)==data["y_test"]).float().mean().item()
    report = dict(train_loss=train_loss,test_accuracy=test_accuracy,
                  parameter_count=sum(p.numel() for p in net.parameters()))
    return net,history,report

data609 = load_classification_data()
net609,history609,report609 = fit_classification(data609)
print("Exercise 2 synthetic-interface baseline:",report609)


### 解析与点评

参数量4100+10100+1010=15210。合成信号只是让接口、训练和比较可运行，不是重建原MNIST1D数据。默认不用网络、不下载、不执行陌生pickle；可读可信字典的适配器不意味着已成功读取不存在的原作业文件。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert data609["x_train"].shape == (320,40) and data609["x_test"].shape == (120,40)
assert data609["y_train"].dtype == torch.long and set(data609["y_train"].tolist()) == set(range(10))
assert "SYNTHETIC" in data609["origin"] and "NOT MNIST1D" in data609["origin"]
assert net609(torch.zeros(1,40)).shape == (1,10)
assert report609["parameter_count"] == 15210
assert len(history609) == 100 and np.isfinite(history609).all()
assert np.isfinite(report609["train_loss"]) and 0 <= report609["test_accuracy"] <= 1
import tempfile
with tempfile.TemporaryDirectory() as temp609:
    bad = Path(temp609)/"not-trusted.data"
    bad.write_bytes(b"not pickle")
    try:
        load_classification_data(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("untrusted pickle was not rejected")

print("自检通过")


## Q610 Homework03 Exercise 2：完整单因素对照入口

**作业改编，精确来源** [[课程/深度学习/作业/Homework03/Homework03.md]] Exercise 2 (3)—(6)。学习期扩展，不是30分钟卷必做的庞大实验。

实现 `comparison_configs()` 与 `run_class_comparisons(data=None,steps=30,lr=.03)`，与Q510相同12种唯一单因素配置：depth=2/3/4/5；width=50/100/150/200；sigmoid/softplus/ReLU；SGD/Adagrad/RMSprop/Adam。
固定数据、初始化seed、更新预算；**所有优化器同学习率**。每配置重新建立优化器，不复用状态。
返回表每行factor/value/完整配置/lr/steps/train_loss/test_accuracy/parameter_count；保存output/Q610-comparison.json，画测试准确率柱图。
默认合成320/120而非原MNIST1D4000/1000、每配置30步CPU。结果只供流程与受控对照，不保证性能，尤其深层sigmoid可能训练不足。若按此测试结果调参，应另留最终测试集；实际研究用验证集选择超参。

**自检接口／需保存的结果**

将默认`steps=30,lr=.03`完整比较的返回值保存为`rows610`：长度12的Python字典列表，第一行为共享基准，随后各行仅改变一个配置因素。每行包含：

- `factor`、`value`：变化因素名称及选项；`origin`：数据来源字符串。
- `depth`、`width`（int），`activation`（`ReLU`、`sigmoid`、`softplus`之一），`optimizer_name`（`SGD`、`Adagrad`、`RMSprop`、`Adam`之一）。
- `steps`（int）、`lr`（float）、`parameter_count`（int），以及该配置最终的`train_loss`、`test_accuracy`（Python float；准确率使用0—1比例）。


<details><summary>提示（先尝试再展开）</summary>

配置表可与回归共享，但拟合函数与评估指标不同：此处是CE与accuracy，不是RMSE。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def quadrant_labels(x):
    if x.ndim != 2 or x.shape[1] != 2 or not torch.isfinite(x).all():
        raise ValueError("X must be finite with shape (N,2)")
    y = torch.empty(len(x), dtype=torch.long)
    y[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    y[(x[:,0]<0)&(x[:,1]>=0)] = 2
    y[(x[:,0]<0)&(x[:,1]<0)] = 3
    y[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return y

class Classifier6(nn.Module):
    def __init__(self, in_dim=2, classes=4, depth=3, width=10, activation="ReLU"):
        super().__init__()
        if depth < 2:
            raise ValueError("depth >= 2")
        activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
        layers, previous = [], in_dim
        for _ in range(depth-1):
            layers += [nn.Linear(previous,width), activations[activation]()]
            previous = width
        self.features = nn.Sequential(*layers)
        self.output = nn.Linear(width, classes)
    def forward(self, x):
        return self.output(self.features(x))  # raw logits, NO softmax here

def fit_quadrants6(steps=100, seed=607):
    torch.manual_seed(seed)
    xt, xs = torch.randn(512,2), torch.randn(192,2)
    yt, ys = quadrant_labels(xt)-1, quadrant_labels(xs)-1
    net = Classifier6(width=16)
    optimizer = torch.optim.Adam(net.parameters(),lr=.02)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(xt),yt)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        logits = net(xs)
        pred0 = logits.argmax(1)
        train_loss = loss_fn(net(xt),yt).item()
        train_acc = (net(xt).argmax(1)==yt).float().mean().item()
        test_acc = (pred0==ys).float().mean().item()
    report = dict(train_loss=train_loss,train_accuracy=train_acc,test_accuracy=test_acc)
    return net, history, (xt,yt,xs,ys), report

def load_classification_data(path=None, *, trusted_pickle=False, n_train=320, n_test=120, seed=609):
    if path is not None and Path(path).is_file():
        if not trusted_pickle:
            raise ValueError("pickle can execute code; explicit trusted opt-in required")
        import pickle
        with Path(path).open("rb") as f:
            raw = pickle.load(f)
        origin = "TRUSTED local pickle: " + str(Path(path).resolve())
    else:
        g = torch.Generator().manual_seed(seed)
        t = torch.linspace(0, 2*np.pi, 40)
        classes = torch.arange(10).reshape(-1,1)
        prototypes = torch.sin((classes+1)*t)+.3*torch.cos((classes+2)*t)
        def sample(n):
            y = torch.arange(n) % 10
            y = y[torch.randperm(n,generator=g)]
            x = prototypes[y] + .15*torch.randn(n,40,generator=g)
            return x, y
        xt,yt = sample(n_train)
        xs,ys = sample(n_test)
        raw = dict(x_train=xt,y_train=yt,x_test=xs,y_test=ys)
        origin = "SYNTHETIC 40-feature/10-class signals; NOT MNIST1D or original homework .data"
    data = {}
    for split in ("train","test"):
        x = torch.as_tensor(raw["x_"+split],dtype=torch.float32).clone()
        raw_y = torch.as_tensor(raw["y_"+split]).reshape(-1)
        if not torch.isfinite(raw_y).all() or not torch.equal(raw_y,raw_y.round()):
            raise ValueError("integer-valued finite labels required")
        y = raw_y.long().clone()
        if x.ndim != 2 or x.shape[1] != 40 or len(x) != len(y) or len(y) == 0:
            raise ValueError("expected X=(N,40), y=(N,)")
        if not torch.isfinite(x).all() or y.min()<0 or y.max()>9:
            raise ValueError("finite features, labels 0..9 required")
        data["x_"+split],data["y_"+split] = x,y
    data["origin"] = origin
    print("Data source:",origin)
    return data

def fit_classification(data, *, depth=3, width=100, activation="ReLU", optimizer_name="SGD",
                       lr=.03, steps=100, seed=609):
    if steps < 1:
        raise ValueError("steps > 0 required")
    torch.manual_seed(seed)
    net = Classifier6(40,10,depth,width,activation)
    optimizers = {"SGD":torch.optim.SGD,"Adagrad":torch.optim.Adagrad,
                  "RMSprop":torch.optim.RMSprop,"Adam":torch.optim.Adam}
    optimizer = optimizers[optimizer_name](net.parameters(),lr=lr)
    loss_fn, history = nn.CrossEntropyLoss(), []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(net(data["x_train"]),data["y_train"])
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        train_loss = loss_fn(net(data["x_train"]),data["y_train"]).item()
        test_accuracy = (net(data["x_test"]).argmax(1)==data["y_test"]).float().mean().item()
    report = dict(train_loss=train_loss,test_accuracy=test_accuracy,
                  parameter_count=sum(p.numel() for p in net.parameters()))
    return net,history,report

def comparison_configs():
    base = dict(depth=3, width=100, activation="ReLU", optimizer_name="SGD")
    rows = [("baseline", "default", base.copy())]
    factors = {"depth": (2, 4, 5), "width": (50, 150, 200),
               "activation": ("sigmoid", "softplus"),
               "optimizer_name": ("Adagrad", "RMSprop", "Adam")}
    for factor, values in factors.items():
        for value in values:
            cfg = base.copy()
            cfg[factor] = value
            rows.append((factor, value, cfg))
    return rows

import json
def run_class_comparisons(data=None, steps=30, lr=.03):
    if data is None:
        data = load_classification_data()
    result = []
    for factor,value,config in comparison_configs():
        _,_,report = fit_classification(data,**config,lr=lr,steps=steps,seed=610)
        result.append(dict(factor=factor,value=value,origin=data["origin"],**config,lr=lr,steps=steps,**report))
    return result

rows610 = run_class_comparisons()
Path("output").mkdir(exist_ok=True)
Path("output/Q610-comparison.json").write_text(json.dumps(rows610,indent=2),encoding="utf-8")
fig,ax = plt.subplots(figsize=(11,4))
ax.bar(range(len(rows610)),[r["test_accuracy"] for r in rows610])
ax.set_xticks(range(len(rows610)),[r["factor"]+":"+str(r["value"]) for r in rows610],rotation=65,ha="right")
ax.set(ylabel="Test accuracy",ylim=(0,1),title="Synthetic signals: single-factor comparison")
fig.tight_layout()
fig.savefig("output/Q610.png",dpi=120)
plt.close(fig)
for row in rows610:
    print(row["factor"],row["value"],"test accuracy",round(row["test_accuracy"],4))


### 解析与点评

同seed在不同架构下不表示相同权重；同结构的优化器对照则能从相同参数起步。固定步数与固定算力也不等价：更宽/更深单步更贵。表中表现只说明该轻量预算和合成分布下的观察。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert len(rows610) == 12
assert {r["depth"] for r in rows610} == {2,3,4,5}
assert {r["width"] for r in rows610} == {50,100,150,200}
assert {r["activation"] for r in rows610} == {"ReLU","sigmoid","softplus"}
assert {r["optimizer_name"] for r in rows610} == {"SGD","Adagrad","RMSprop","Adam"}
base610,keys610 = rows610[0],["depth","width","activation","optimizer_name"]
assert all(sum(r[k]!=base610[k] for k in keys610)==1 for r in rows610[1:])
assert all(r["lr"]==.03 and r["steps"]==30 for r in rows610)
assert all(np.isfinite(r["train_loss"]) and 0<=r["test_accuracy"]<=1 for r in rows610)
assert Path("output/Q610-comparison.json").is_file()

print("自检通过")
